# Bank Marketing — Data Preprocessing

## 1. Introduction

이 노트북에서는 모델 개발에 사용할 baseline 전처리 구성을 구현한다.

- `duration`을 제외하고 나머지 19개 원본 예측 변수를 유지한다.
- `pdays = 999`의 sentinel 의미를 명시적으로 표현한다.
- 기존의 시간 순서에 따른 Train/Validation 분할을 유지한다.
- 전처리는 Train에서만 fit한다.
- Test는 최종 평가를 위해 보류한다.


## 2. Load Data & Splits

원본 행 순서로 데이터를 읽고 기존의 시간 순서에 따른 분할 경계를 재현한다.
이 노트북에서는 Train과 Validation만 준비한다.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "bank-additional"
    / "bank-additional-full.csv"
)

df = pd.read_csv(RAW_DATA_PATH, sep=";")

train_end = int(len(df) * 0.70)
valid_end = int(len(df) * 0.90)

train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()

In [2]:
target = "y"
excluded_features = ["duration"]
modeling_features = [
    column for column in df.columns
    if column not in [target, *excluded_features]
]

X_train = train_df[modeling_features].copy()
X_valid = valid_df[modeling_features].copy()

target_mapping = {"no": 0, "yes": 1}
y_train = train_df[target].map(target_mapping)
y_valid = valid_df[target].map(target_mapping)

pd.DataFrame(
    {
        "rows": [len(X_train), len(X_valid)],
        "features": [X_train.shape[1], X_valid.shape[1]],
        "positive_rate": [y_train.mean(), y_valid.mean()],
    },
    index=["Train", "Validation"],
)

,rows,features,positive_rate
Train,28831,19,0.055704
Validation,8238,19,0.138626


## 3. Feature Definition

필요한 전처리 방식에 따라 feature 그룹을 명시적으로 정의한다.
`pdays`는 19개 원본 입력 중 하나로 유지하되, scaling 전에 sentinel 여부와 경과 일수를 분리한다.


In [3]:
numeric_features = [
    "age",
    "campaign",
    "previous",
    "emp.var.rate",
    "cons.price.idx",
    "cons.conf.idx",
    "euribor3m",
    "nr.employed",
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]

special_features = ["pdays"]

pd.Series(
    {
        "numeric": len(numeric_features),
        "categorical": len(categorical_features),
        "sentinel_aware": len(special_features),
        "total_raw": len(numeric_features + categorical_features + special_features),
    },
    name="feature_count",
)

numeric            8
categorical       10
sentinel_aware     1
total_raw         19
Name: feature_count, dtype: int64

## 4. Preprocessing Strategy

| Feature 그룹 | Baseline 변환 |
|---|---|
| 일반 수치형 feature 8개 | `StandardScaler` |
| `pdays` | 이전 연락 여부와 경과 일수로 분리한 뒤 `StandardScaler` |
| 범주형 feature 10개 | `OneHotEncoder(handle_unknown="ignore")` |

`pdays = 999`이면 `pdays_previously_contacted = 0`, `pdays_elapsed = 0`으로 변환한다.
이전 연락이 기록된 레코드는 indicator를 `1`로 설정하고 관측된 경과 일수를 유지한다.
원본의 `999`를 일반적인 경과 일수로 취급하지 않는다.

Baseline은 `"unknown"`을 범주로 유지한다. 결측값 대체, clipping, 희소 범주 통합,
추가 feature engineering은 적용하지 않는다.


## 5. Preprocessing Pipeline

### 5.1 Build Preprocessor

`ColumnTransformer`로 각 변환을 지정된 열에만 적용한다.


In [4]:
def split_pdays(values):
    pdays = np.asarray(values).reshape(-1)
    previously_contacted = (pdays != 999).astype(float)
    elapsed_days = np.where(pdays == 999, 0.0, pdays.astype(float))
    return np.column_stack([previously_contacted, elapsed_days])


def pdays_feature_names(transformer, input_features):
    return np.asarray(
        ["pdays_previously_contacted", "pdays_elapsed"],
        dtype=object,
    )


pdays_transformer = Pipeline(
    steps=[
        (
            "derive",
            FunctionTransformer(
                split_pdays,
                validate=False,
                feature_names_out=pdays_feature_names,
            ),
        ),
        ("scale", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        ("pdays", pdays_transformer, special_features),
    ],
    remainder="drop",
)

### 5.2 Fit on Train & Transform Validation

Scaler의 통계와 encoder의 범주는 Train에서 학습한다.
Validation에는 재학습 없이 같은 전처리기를 적용한다.


In [5]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_valid_prepared = preprocessor.transform(X_valid)

In [6]:
feature_names = preprocessor.get_feature_names_out()

transformation_summary = pd.DataFrame(
    {
        "input_shape": [X_train.shape, X_valid.shape],
        "output_shape": [X_train_prepared.shape, X_valid_prepared.shape],
        "output_type": [
            type(X_train_prepared).__name__,
            type(X_valid_prepared).__name__,
        ],
    },
    index=["Train", "Validation"],
)

display(transformation_summary)
pd.Series(feature_names, name="transformed_feature")

,input_shape,output_shape,output_type
Train,"(28831, 19)","(28831, 62)",ndarray
Validation,"(8238, 19)","(8238, 62)",ndarray


0                              num__age
1                         num__campaign
2                         num__previous
3                     num__emp.var.rate
4                   num__cons.price.idx
                    ...                
57                cat__poutcome_failure
58            cat__poutcome_nonexistent
59                cat__poutcome_success
60    pdays__pdays_previously_contacted
61                 pdays__pdays_elapsed
Name: transformed_feature, Length: 62, dtype: str

Baseline 전처리 결과는 총 62개 feature이다. Scaling한 수치형 열 8개,
Train에서 학습한 one-hot 열 52개, sentinel을 분리한 `pdays` 열 2개로 구성된다.


## 6. Preprocessing Summary

- 예측 시점에 사용할 수 있는 원본 예측 변수 19개를 유지하고 `duration`은 제외한다.
- 일반 수치형 feature 8개에는 `StandardScaler`를 적용한다.
- 원본 `pdays`를 이전 연락 여부와 경과 일수로 대체하고, sentinel을 처리한 뒤 두 열을 scaling한다.
- 범주형 feature에는 `OneHotEncoder(handle_unknown="ignore")`를 적용한다.
- 전처리기는 Train에서 fit하고 Validation에 그대로 적용한다.
- Baseline 표현에는 결측값 대체, clipping, 범주 통합, 추가 feature engineering을 적용하지 않는다.

이 표현은 EDA에서 확인한 `pdays = 999`의 의미를 따른다. 해당 표현의 성능은
baseline과 모델 실험 노트북에서 다시 확인해야 하며,
이전의 원본 `pdays` 기반 지표를 수정된 구성의 성능 근거로 사용할 수 없다.
